# Agent Fundamentals

**Module:** 10-agentic-ai-concepts

**Notebook:** `01-agent-fundamentals.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **What is an Agent?** with clear contracts and failure modes
- Explain and apply **Brain** with clear contracts and failure modes
- Explain and apply **Memory** with clear contracts and failure modes
- Explain and apply **Tools** with clear contracts and failure modes
- Explain and apply **Planning & Execution** with clear contracts and failure modes
- Explain and apply **Lifecycle** with clear contracts and failure modes
- Explain and apply **Minimal Agent Loop** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — Agent Fundamentals

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **What is an Agent?**
2. **Brain**
3. **Memory**
4. **Tools**
5. **Planning & Execution**
6. **Lifecycle**
7. **Minimal Agent Loop**

Read top-to-bottom once, then revisit weak spots with the exercises.


## What is an Agent?

### Definition
An **agent** is a system that uses a model to pursue a goal by choosing actions (often tools) over multiple steps, guided by memory and policies.

### Why it matters
Single-shot chat cannot operate software, fetch live state, or adapt when subgoals fail.

### How it works
Loop: observe → reason → act → observe… with stop conditions, permissions, and audit logs.

### Intuition
A junior operator with a runbook and a limited toolbelt—not an omniscient oracle.

### Pitfalls
- Unbounded loops
- Tools without authz
- No success/failure criteria

### When to use
Multi-step goals needing tools, not pure one-off Q&A.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does What is an Agent? improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "What is an Agent?" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "What is an Agent?"
    notebook: str = "01-agent-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
from enum import Enum

class Phase(Enum):
    PLAN = "plan"
    ACT = "act"
    CHECK = "check"
    DONE = "done"

def agent_loop(goal: str, max_steps=3):
    log = []
    phase = Phase.PLAN
    for step in range(max_steps):
        if phase == Phase.PLAN:
            plan = [f"step{i+1}" for i in range(2)]
            log.append(("plan", plan)); phase = Phase.ACT
        elif phase == Phase.ACT:
            log.append(("act", f"execute {goal}")); phase = Phase.CHECK
        elif phase == Phase.CHECK:
            log.append(("check", "ok")); phase = Phase.DONE
        else:
            break
    return log

print(agent_loop("triage ticket"))


In [ ]:
# Supervisor routes to specialists
def route_intent(text: str) -> str:
    t = text.lower()
    if "refund" in t or "invoice" in t: return "billing_agent"
    if "login" in t or "sso" in t: return "auth_agent"
    return "general_agent"

for q in ["refund please", "SSO loop", "hello"]:
    print(q, "->", route_intent(q))


## Brain

### Definition
**Brain** is a core building block in 01-agent-fundamentals within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Brain typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Brain: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Brain as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Brain as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Brain
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Brain when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Brain" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Brain"
    notebook: str = "01-agent-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Brain"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Brain"}
strong = {"definition": "Brain", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Brain"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Brain", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Brain

**Situation:** A team wants to productionize a feature involving **Brain**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Memory

### Definition
**Memory** is a core building block in 01-agent-fundamentals within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Memory typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Memory: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Memory as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Memory as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Memory
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Memory when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Memory" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Memory"
    notebook: str = "01-agent-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
from collections import deque

class MemoryStore:
    def __init__(self, k=4):
        self.short = deque(maxlen=k)
        self.summary = ""
        self.episodic = []

    def add_turn(self, role, content):
        self.short.append({"role": role, "content": content})
        if role == "user":
            self.episodic.append(content[:160])

    def pack(self):
        return {"summary": self.summary, "recent": list(self.short), "episodes": self.episodic[-5:]}

mem = MemoryStore()
mem.add_turn("user", "My plan is Pro")
mem.add_turn("assistant", "Noted: plan=Pro")
mem.summary = "User on Pro plan"
print(mem.pack())


In [ ]:
# Semantic memory stub: bag-of-words retrieval
DOCS = ["refund policy under $5", "SSO allowlist redirects", "P0 outage page oncall"]

def retrieve(q: str, k=2):
    qw = set(q.lower().split())
    scored = sorted(DOCS, key=lambda d: len(qw & set(d.split())), reverse=True)
    return scored[:k]

print(retrieve("need refund for small charge"))


## Tools

### Definition
**Tools** is a core building block in 01-agent-fundamentals within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Tools typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Tools: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Tools as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Tools as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Tools
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Tools when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Tools" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Tools"
    notebook: str = "01-agent-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


### Worked scenario — Tools

**Situation:** A team wants to productionize a feature involving **Tools**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Planning & Execution

### Definition
**Planning & Execution** is a core building block in 01-agent-fundamentals within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Planning & Execution typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Planning & Execution: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Planning & Execution as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Planning & Execution as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Planning & Execution
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Planning & Execution when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Planning & Execution" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Planning & Execution"
    notebook: str = "01-agent-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
from enum import Enum

class Phase(Enum):
    PLAN = "plan"
    ACT = "act"
    CHECK = "check"
    DONE = "done"

def agent_loop(goal: str, max_steps=3):
    log = []
    phase = Phase.PLAN
    for step in range(max_steps):
        if phase == Phase.PLAN:
            plan = [f"step{i+1}" for i in range(2)]
            log.append(("plan", plan)); phase = Phase.ACT
        elif phase == Phase.ACT:
            log.append(("act", f"execute {goal}")); phase = Phase.CHECK
        elif phase == Phase.CHECK:
            log.append(("check", "ok")); phase = Phase.DONE
        else:
            break
    return log

print(agent_loop("triage ticket"))


In [ ]:
# Supervisor routes to specialists
def route_intent(text: str) -> str:
    t = text.lower()
    if "refund" in t or "invoice" in t: return "billing_agent"
    if "login" in t or "sso" in t: return "auth_agent"
    return "general_agent"

for q in ["refund please", "SSO loop", "hello"]:
    print(q, "->", route_intent(q))


## Lifecycle

### Definition
**Lifecycle** is a core building block in 01-agent-fundamentals within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Lifecycle typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Lifecycle: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Lifecycle as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Lifecycle as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Lifecycle
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Lifecycle when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Lifecycle" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Lifecycle"
    notebook: str = "01-agent-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Lifecycle"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Lifecycle"}
strong = {"definition": "Lifecycle", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Lifecycle"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Lifecycle", "passed": len(checks)-len(failed), "failed": failed})


In [ ]:
# Demo: decision table for applying "Lifecycle"
options = [
    {"option": "baseline_simple", "quality": 0.7, "cost": 1, "ops": 0.9},
    {"option": "advanced_lifecycle", "quality": 0.85, "cost": 3, "ops": 0.6},
]
for o in options:
    o["utility"] = round(o["quality"] * 2 - 0.3*o["cost"] + 0.5*o["ops"], 3)
best = max(options, key=lambda x: x["utility"])
print("ranked:", sorted(options, key=lambda x: -x["utility"]))
print("prefer:", best["option"])


### Worked scenario — Lifecycle

**Situation:** A team wants to productionize a feature involving **Lifecycle**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Minimal Agent Loop

### Definition
**Minimal Agent Loop** is a core building block in 01-agent-fundamentals within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Minimal Agent Loop typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Minimal Agent Loop: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Minimal Agent Loop as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Minimal Agent Loop as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Minimal Agent Loop
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Minimal Agent Loop when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Minimal Agent Loop" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Minimal Agent Loop"
    notebook: str = "01-agent-fundamentals"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_6 = ConceptContract()
print(json.dumps({"contract": asdict(contract_6), "health": contract_6.health()}, indent=2))


In [ ]:
from enum import Enum

class Phase(Enum):
    PLAN = "plan"
    ACT = "act"
    CHECK = "check"
    DONE = "done"

def agent_loop(goal: str, max_steps=3):
    log = []
    phase = Phase.PLAN
    for step in range(max_steps):
        if phase == Phase.PLAN:
            plan = [f"step{i+1}" for i in range(2)]
            log.append(("plan", plan)); phase = Phase.ACT
        elif phase == Phase.ACT:
            log.append(("act", f"execute {goal}")); phase = Phase.CHECK
        elif phase == Phase.CHECK:
            log.append(("check", "ok")); phase = Phase.DONE
        else:
            break
    return log

print(agent_loop("triage ticket"))


In [ ]:
# Supervisor routes to specialists
def route_intent(text: str) -> str:
    t = text.lower()
    if "refund" in t or "invoice" in t: return "billing_agent"
    if "login" in t or "sso" in t: return "auth_agent"
    return "general_agent"

for q in ["refund please", "SSO loop", "hello"]:
    print(q, "->", route_intent(q))


## Comparison Snapshot

Use this table when reviewing designs in **Agent Fundamentals**.

| Topic | Do | Don't |
|-------|----|-------|
| What is an Agent? | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Brain | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Memory | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Tools | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Planning & Execution | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Lifecycle | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| What is an Agent? | Key concept covered in this notebook; see its section for definition and pitfalls |
| Brain | Key concept covered in this notebook; see its section for definition and pitfalls |
| Memory | Key concept covered in this notebook; see its section for definition and pitfalls |
| Tools | Key concept covered in this notebook; see its section for definition and pitfalls |
| Planning & Execution | Key concept covered in this notebook; see its section for definition and pitfalls |
| Lifecycle | Key concept covered in this notebook; see its section for definition and pitfalls |
| Minimal Agent Loop | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **Agent Fundamentals** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **10-agentic-ai-concepts**.


## Try It Yourself

1. Implement a failing test/fixture for **What is an Agent?**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Brain**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Memory**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Tools**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Planning & Execution**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
